# 

In [1]:


# Import all the libraries used in this notebook
# from pvlib.solarposition import get_solarposition
import datetime
import os
from glob import glob
from os.path import join

from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected=True)

import plotly
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
from  plotly.graph_objs import *
import plotly.io as pio
pio.renderers.default = 'iframe'

print(os.environ["CONDA_DEFAULT_ENV"])  # Check the name of the current Conda environment

/glade/work/dettling/conda-envs/plotlyenv


In [7]:
#
# Define this time series layout once, will use again in time series plots in rest of notebook
#
timeSeriesLayout = dict(
    title="Time Series with Rangeslider",
    xaxis=dict(
        rangeselector=dict(
            buttons=list(
                [
                    dict(count=1, label="1d", step="day", stepmode="forward"),
                    dict(count=7, label="1w", step="week", stepmode="forward"),
                    dict(count=30, label="1m", step="month", stepmode="forward"),
                    dict(step="all"),
                ]
            )
        ),
        rangeslider=dict(),
        type="date",
    ),
    width=1200,  # Custom width in pixels
    height=800,  # Custom height in pixels
)

layout = timeSeriesLayout
   
def drawTraces(*args):
    data = [*args]
    plotly.offline.iplot(data, filename="basic-line-plot")
    layout = timeSeriesLayout


In [11]:
# mvco= pd.read_csv('mvco_mlsl_qc.csv')
rvsr = pd.read_csv("/glade/campaign/ral/wsap/oracleMLSL/data/RVSR_MAPS_FLIP.20170930-20171022.10min.csv", encoding="utf-8", delimiter=",")

rvsr["Time"] = pd.to_datetime(rvsr["Time"])
rvsr.index = rvsr["Time"]
print(len(rvsr.columns))
print(len(rvsr.index))
for col in rvsr.columns:
    print(col)

194
3311
index
Time
RVSR_temperature:12.54_m:C
RVSR_wind_speed:13.49_m:m/s
RVSR_wind_direction:13.49_m:deg
RVSR_water_surface_temperature:-3.5_m:C
RVSR_pressure:0_m:mb
RVSR_relative_humidity:12.54_m:%
RVSR_solar_downwelling_flux:?:W/m2
RVSR_IR_downwelling_flux:?:W/m2
RVSR_COARE_sensible_heat_flux:?:W/m2
RVSR_COARE_latent_heat_flux:?:W/m2
RVSR_COARE_Obukhov_length_scale:?:m
RVSR_COARE_Ustar:?:m/s
RVSR_wu_NOAA_sonic:13.49_m:m/s
RVSR_wv_NOAA_sonic:13.49_m:m/s
RVSR_wt_NOAA_sonic:13.49_m:m/s
RVSR_wq_NOAA_sonic:13.49_m:m/s
RVSR_wu_NDS1_sonic:11.54_m:m/s
RVSR_wv_NDS1_sonic:11.54_m:m/s
RVSR_wt_NDS1_sonic:11.54_m:m/s
RVSR_wu_NDS0_sonic:9.69_m:m/s
RVSR_wv_NDS0_sonic:9.69_m:m/s
RVSR_wt_NDS0_sonic:9.69_m:m/s
RVSR_sensible_heat_covariance_NOAA_sonic:13.49_m:W/m2
RVSR_sensible_heat_ID_NOAA_sonic:13.49_m:W/m2
RVSR_latent_heat_covariance_NOAA_sonic:13.49_m:W/m2
RVSR_latent_heat_ID_NOAA_sonic:13.49_m:W/m2
RVSR_sensible_heat_covariance_NDS1_sonic:11.54_m:W/m2
RVSR_sensible_heat_ID_NDS1_sonic:11.54_m:W/m

In [74]:

rvsr.columns = rvsr.columns.str.replace('velicity', 'velocity')
flip_ustar_cols = [col for col in rvsr.columns if 'friction_velocity' in col]
print( rvsr[flip_ustar_cols].columns)

Index(['FLIP_friction_velocity:4.69_m:m/s',
       'FLIP_filtered_friction_velocity:4.69_m:m/s',
       'FLIP_friction_velocity:5.55_m:m/s',
       'FLIP_filtered_friction_velocity:5.55_m:m/s',
       'FLIP_friction_velocity:6.82_m:m/s',
       'FLIP_filtered_friction_velocity:6.82_m:m/s',
       'FLIP_friction_velocity:7.93_m:m/s',
       'FLIP_filtered_friction_velocity:7.93_m:m/s',
       'FLIP_friction_velocity:10.55_m:m/s',
       'FLIP_filtered_friction_velocity:10.55_m:m/s',
       'FLIP_friction_velocity:11.95_m:m/s',
       'FLIP_filtered_friction_velocity:11.95_m:m/s',
       'FLIP_friction_velocity:13.73_m:m/s',
       'FLIP_filtered_friction_velocity:13.73_m:m/s'],
      dtype='object')


In [19]:
flip_flux_cols = [col for col in flip.columns if 'sensible_heat_flux' in col]
print( flip[flip_flux_cols].columns)

Index(['FLIP_sensible_heat_flux:4.69_m:W/m2',
       'FLIP_filtered_sensible_heat_flux:4.69_m:W/m2',
       'FLIP_sensible_heat_flux:5.55_m:W/m2',
       'FLIP_filtered_sensible_heat_flux:5.55_m:W/m2',
       'FLIP_sensible_heat_flux:6.82_m:W/m2',
       'FLIP_filtered_sensible_heat_flux:6.82_m:W/m2',
       'FLIP_sensible_heat_flux:7.93_m:W/m2',
       'FLIP_filtered_sensible_heat_flux:7.93_m:W/m2',
       'FLIP_sensible_heat_flux:10.55_m:W/m2',
       'FLIP_filtered_sensible_heat_flux:10.55_m:W/m2',
       'FLIP_sensible_heat_flux:11.95_m:W/m2',
       'FLIP_filtered_sensible_heat_flux:11.95_m:W/m2',
       'FLIP_sensible_heat_flux:13.73_m:W/m2',
       'FLIP_filtered_sensible_heat_flux:13.73_m:W/m2'],
      dtype='object')


In [92]:
tracea = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_sensible_heat_flux:4.69_m:W/m2'],
         name= 'FLIP_sensible_heat_flux:4.69_m:W/m2',
         line=dict(color="purple"),
         connectgaps=False,
         opacity=0.5,
)

traceb = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_filtered_sensible_heat_flux:4.69_m:W/m2'],
         name= 'FLIP_filtered_sensible_heat_flux:4.69_m:W/m2',
         line=dict(color="blue"),
         connectgaps=False,
         opacity=0.5,
)
tracec = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_sensible_heat_flux:5.55_m:W/m2'],
         name= 'FLIP_sensible_heat_flux:5.55_m:W/m2',
         line=dict(color="cyan"),
         connectgaps=False,
         opacity=0.5,
)

traced = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_filtered_sensible_heat_flux:5.55_m:W/m2'],
         name= 'FLIP_filtered_sensible_heat_flux:5.55_m:W/m2',
         line=dict(color="lightblue"),
         connectgaps=False,
         opacity=0.5,
)

tracee = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_sensible_heat_flux:6.82_m:W/m2'],
         name= 'FLIP_sensible_heat_flux:6.82_m:W/m2',
         line=dict(color="orange"),
         connectgaps=False,
         opacity=0.5,
)

tracef = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_filtered_sensible_heat_flux:6.82_m:W/m2'],
         name= 'FLIP_filtered_sensible_heat_flux:6.82_m:W/m2',
         line=dict(color="gold"),
         connectgaps=False,
         opacity=0.5,
)
traceg = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_sensible_heat_flux:7.93_m:W/m2'],
         name= 'FLIP_sensible_heat_flux:7.93_m:W/m2',
         line=dict(color="green"),
         connectgaps=False,
         opacity=0.5,
)

traceh = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_filtered_sensible_heat_flux:7.93_m:W/m2'],
         name= 'FLIP_filtered_sensible_heat_flux:7.93_m:W/m2',
         line=dict(color="SpringGreen"),
         connectgaps=False,
         opacity=0.5,
)
tracei = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_sensible_heat_flux:10.55_m:W/m2'],
         name= 'FLIP_sensible_heat_flux:10.55_m:W/m2',
         line=dict(color="sienna"),
         connectgaps=False,
         opacity=0.5,
)

tracej = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_filtered_sensible_heat_flux:10.55_m:W/m2'],
         name= 'FLIP_filtered_sensible_heat_flux:10.55_m:W/m2',
         line=dict(color="brown"),
         connectgaps=False,
         opacity=0.5,
)
tracek = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_sensible_heat_flux:11.95_m:W/m2'],
         name= 'FLIP_sensible_heat_flux:11.95_m:W/m2',
         line=dict(color="darkslategray"),
         connectgaps=False,
         opacity=0.5,
)

tracel = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_filtered_sensible_heat_flux:11.95_m:W/m2'],
         name= 'FLIP_filtered_sensible_heat_flux:11.95_m:W/m2',
         line=dict(color="black"),
         connectgaps=False,
         opacity=0.5,
)

tracem = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_sensible_heat_flux:13.73_m:W/m2'],
         name= 'FLIP_sensible_heat_flux:13.73_m:W/m2',
         line=dict(color="maroon"),
         connectgaps=False,
         opacity=0.5,
)

tracen = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_filtered_sensible_heat_flux:13.73_m:W/m2'],
         name= 'FLIP_filtered_sensible_heat_flux:13.73_m:W/m2',
         line=dict(color="red"),
         connectgaps=False,
         opacity=0.5,
)

In [93]:

# we want filtered
drawTraces(*[traceb, traced, tracef, traceh, tracej, tracel, tracen ])

In [94]:
drawTraces(*[traceb, traced, tracef, traceh, tracej, tracel , tracen])

In [61]:
traceaa = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_sensible_heat_flux:4.69_m:W/m2']/4.69,
         name= 'FLIP_sensible_heat_flux:4.69_m:W/m2',
         line=dict(color="purple"),
         connectgaps=False,
         opacity=0.5,
)

tracebb = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_filtered_sensible_heat_flux:4.69_m:W/m2']/4.69,
         name= 'FLIP_filtered_sensible_heat_flux:4.69_m:W/m2',
         line=dict(color="blue"),
         connectgaps=False,
         opacity=0.5,
)
tracecc = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_sensible_heat_flux:5.55_m:W/m2']/5.55,
         name= 'FLIP_sensible_heat_flux:5.55_m:W/m2',
         line=dict(color="cyan"),
         connectgaps=False,
         opacity=0.5,
)

tracedd = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_filtered_sensible_heat_flux:5.55_m:W/m2']/5.55,
         name= 'FLIP_filtered_sensible_heat_flux:5.55_m:W/m2',
         line=dict(color="lightblue"),
         connectgaps=False,
         opacity=0.5,
)

traceee = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_sensible_heat_flux:6.82_m:W/m2']/6.82,
         name= 'FLIP_sensible_heat_flux:6.82_m:W/m2',
         line=dict(color="orange"),
         connectgaps=False,
         opacity=0.5,
)

traceff = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_filtered_sensible_heat_flux:6.82_m:W/m2']/6.82,
         name= 'FLIP_filtered_sensible_heat_flux:6.82_m:W/m2',
         line=dict(color="gold"),
         connectgaps=False,
         opacity=0.5,
)
traceg = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_sensible_heat_flux:7.93_m:W/m2']/7.93,
         name= 'FLIP_sensible_heat_flux:7.93_m:W/m2',
         line=dict(color="green"),
         connectgaps=False,
         opacity=0.5,
)

tracehh = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_filtered_sensible_heat_flux:7.93_m:W/m2']/7.93,
         name= 'FLIP_filtered_sensible_heat_flux:7.93_m:W/m2',
         line=dict(color="SpringGreen"),
         connectgaps=False,
         opacity=0.5,
)
traceii = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_sensible_heat_flux:10.55_m:W/m2']/10.55,
         name= 'FLIP_sensible_heat_flux:10.55_m:W/m2',
         line=dict(color="sienna"),
         connectgaps=False,
         opacity=0.5,
)

tracejj = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_filtered_sensible_heat_flux:10.55_m:W/m2']/10.55,
         name= 'FLIP_filtered_sensible_heat_flux:10.55_m:W/m2',
         line=dict(color="brown"),
         connectgaps=False,
         opacity=0.5,
)
tracekk = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_sensible_heat_flux:11.95_m:W/m2']/11.95,
         name= 'FLIP_sensible_heat_flux:11.95_m:W/m2',
         line=dict(color="darkslategray"),
         connectgaps=False,
         opacity=0.5,
)

tracell = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_filtered_sensible_heat_flux:11.95_m:W/m2']/11.95,
         name= 'FLIP_filtered_sensible_heat_flux:11.95_m:W/m2',
         line=dict(color="black"),
         connectgaps=False,
         opacity=0.5,
)

tracemm = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_sensible_heat_flux:13.73_m:W/m2']/13.73,
         name= 'FLIP_sensible_heat_flux:13.73_m:W/m2',
         line=dict(color="maroon"),
         connectgaps=False,
         opacity=0.5,
)

tracenn = go.Scatter(
         x=rvsr.index,
         y=rvsr['FLIP_filtered_sensible_heat_flux:13.73_m:W/m2']/13.73,
         name= 'FLIP_filtered_sensible_heat_flux:13.73_m:W/m2',
         line=dict(color="red"),
         connectgaps=False,
         opacity=0.5,
)


In [78]:
drawTraces(*[tracebb, tracedd, traceff, tracehh, tracejj, tracell , tracenn])

In [96]:
flip_filtered_cols = [col for col in rvsr.columns if 'FLIP' in col]
flip = rvsr[['index','Time','RVSR_pressure:0_m:mb', 'RVSR_water_surface_temperature:-3.5_m:C' ] + flip_filtered_cols]

In [97]:
print(flip.columns)

Index(['index', 'Time', 'RVSR_pressure:0_m:mb',
       'RVSR_water_surface_temperature:-3.5_m:C', 'FLIP_proftime',
       'FLIP_skin_temperature_at_saturation:0_m:C',
       'FLIP_specific_humidity:?:k/Kg', 'FLIP_pressure:?:mb',
       'FLIP_downwelling_lw_irradiance:?:W/m2',
       'FLIP_downwelling_sw_irradiance:?:W/m2',
       ...
       'FLIP_filtered_sensible_heat_flux:13.73_m:W/m2',
       'FLIP_latent_heat_flux:13.73_m:W/m2',
       'FLIP_filtered_latent_heat_flux:13.73_m:W/m2',
       'FLIP_peak_wave_angular_frequency:0_m:rad/s',
       'FLIP_peak_wave_number:0_m:rad/m',
       'FLIP_avg_wave_angular_frequency:0_m:rad/s',
       'FLIP_avg_wave_number:0_m:rad/m',
       'FLIP_penultimate_wave_angular_frequency:0_m:rad/s',
       'FLIP_penultimate_wave_number:0_m:rad/m',
       'FLIP_significant_wave_height:0_m:m'],
      dtype='object', length=130)


In [6]:


unique_heights = []
for col in flip.columns:
   if ':' in col:
      var,height, units = col.split(':',3)
      if height not in unique_heights:
         unique_heights.append(height)
print(unique_heights)

for h in unique_heights:
    for col in flip.columns:
        if h in col:
            var,height, units = col.split(':',3)
            print(var, " ",  height)

['0_m', '-3.5_m', '?', '2.98_m', '4.03_m', '4.94_m', '5.86_m', '8.93_m', '12.35_m', '15.76_m', '4.69_m', '5.55_m', '6.82_m', '7.93_m', '10.55_m', '11.95_m', '13.73_m']
RVSR_pressure   0_m
FLIP_skin_temperature_at_saturation   0_m
FLIP_peak_wave_angular_frequency   0_m
FLIP_peak_wave_number   0_m
FLIP_avg_wave_angular_frequency   0_m
FLIP_avg_wave_number   0_m
FLIP_penultimate_wave_angular_frequency   0_m
FLIP_penultimate_wave_number   0_m
FLIP_significant_wave_height   0_m
RVSR_water_surface_temperature   -3.5_m
FLIP_specific_humidity   ?
FLIP_pressure   ?
FLIP_downwelling_lw_irradiance   ?
FLIP_downwelling_sw_irradiance   ?
FLIP_wind_dir   ?
FLIP_hr_wind_speed   2.98_m
FLIP_hr_filtered_wind_speed   2.98_m
FLIP_hr_air_tmp   2.98_m
FLIP_hr_specific_humidity   2.98_m
FLIP_hr_wind_speed   4.03_m
FLIP_hr_filtered_wind_speed   4.03_m
FLIP_hr_air_tmp   4.03_m
FLIP_hr_specific_humidity   4.03_m
FLIP_hr_wind_speed   4.94_m
FLIP_hr_filtered_wind_speed   4.94_m
FLIP_hr_air_tmp   4.94_m
FLIP_hr_s

In [98]:
keep_heights = ['-3.5_m','0_m', '?','12.35_m', '15.76_m','13.73_m']
keep_cols = []
for col in flip.columns:
    if ':' in col:
        var,height, units = col.split(':',3)
        if height in keep_heights and 'penultimadte' not in col:
            keep_cols.append(col)
    else:
        if 'penultimate' not in col and 'peak' not in col:
           keep_cols.append(col)
print(keep_cols)

['index', 'Time', 'RVSR_pressure:0_m:mb', 'RVSR_water_surface_temperature:-3.5_m:C', 'FLIP_proftime', 'FLIP_skin_temperature_at_saturation:0_m:C', 'FLIP_specific_humidity:?:k/Kg', 'FLIP_pressure:?:mb', 'FLIP_downwelling_lw_irradiance:?:W/m2', 'FLIP_downwelling_sw_irradiance:?:W/m2', 'FLIP_wind_dir:?:degrees', 'FLIP_hr_wind_speed:12.35_m:m/s', 'FLIP_hr_filtered_wind_speed:12.35_m:m/s', 'FLIP_hr_air_tmp:12.35_m:C', 'FLIP_hr_specific_humidity:12.35_m:g/Kg', 'FLIP_hr_wind_speed:15.76_m:m/s', 'FLIP_hr_filtered_wind_speed:15.76_m:m/s', 'FLIP_hr_air_tmp:15.76_m:C', 'FLIP_hr_specific_humidity:15.76_m:g/Kg', 'FLIP_friction_velocity:13.73_m:m/s', 'FLIP_filtered_friction_velocity:13.73_m:m/s', 'FLIP_wind_stress:13.73_m:N/m2', 'FLIP_filtered_wind_stress:13.73_m:N/m2', 'FLIP_turbulent_temp_scale:13.73_m:C', 'FLIP_filtered_turbulent_temp_scale:13.73_m:C', 'FLIP_turbulent_water_vapor_scale:13.73_m:g/Kg', 'FLIP_filtered_turbulent_water_vapor_scale:13.73_m:g/Kg', 'FLIP_sensible_heat_flux:13.73_m:W/m2',

In [99]:
flipLtd = flip[keep_cols]
flipLtd = flipLtd.drop(['FLIP_proftime', 'index', 'FLIP_penultimate_wave_angular_frequency:0_m:rad/s','FLIP_penultimate_wave_number:0_m:rad/m'], axis = 1)
for col in flipLtd.columns:
    print(col)

Time
RVSR_pressure:0_m:mb
RVSR_water_surface_temperature:-3.5_m:C
FLIP_skin_temperature_at_saturation:0_m:C
FLIP_specific_humidity:?:k/Kg
FLIP_pressure:?:mb
FLIP_downwelling_lw_irradiance:?:W/m2
FLIP_downwelling_sw_irradiance:?:W/m2
FLIP_wind_dir:?:degrees
FLIP_hr_wind_speed:12.35_m:m/s
FLIP_hr_filtered_wind_speed:12.35_m:m/s
FLIP_hr_air_tmp:12.35_m:C
FLIP_hr_specific_humidity:12.35_m:g/Kg
FLIP_hr_wind_speed:15.76_m:m/s
FLIP_hr_filtered_wind_speed:15.76_m:m/s
FLIP_hr_air_tmp:15.76_m:C
FLIP_hr_specific_humidity:15.76_m:g/Kg
FLIP_friction_velocity:13.73_m:m/s
FLIP_filtered_friction_velocity:13.73_m:m/s
FLIP_wind_stress:13.73_m:N/m2
FLIP_filtered_wind_stress:13.73_m:N/m2
FLIP_turbulent_temp_scale:13.73_m:C
FLIP_filtered_turbulent_temp_scale:13.73_m:C
FLIP_turbulent_water_vapor_scale:13.73_m:g/Kg
FLIP_filtered_turbulent_water_vapor_scale:13.73_m:g/Kg
FLIP_sensible_heat_flux:13.73_m:W/m2
FLIP_filtered_sensible_heat_flux:13.73_m:W/m2
FLIP_latent_heat_flux:13.73_m:W/m2
FLIP_filtered_latent_he

In [100]:

traceg = go.Scatter(
         x=flipLtd.index,
         y=flipLtd["FLIP_hr_wind_speed:15.76_m:m/s"],
         name= "FLIP_hr_wind_speed:15.76_m:m/s",
         line=dict(color="blue"),
         connectgaps=False,
         opacity=0.5,
)
traceh =go.Scatter(
         x=flipLtd.index,
         y=flipLtd["FLIP_hr_filtered_wind_speed:15.76_m:m/s"],
         name= "FLIP_hr_filtered_wind_speed:15.76_m:m/s",
         line=dict(color="green"),
         connectgaps=False,
         opacity=0.5,
)
drawTraces(*[traceg, traceh])

In [101]:

traceg = go.Scatter(
         x=flipLtd.index,
         y=flipLtd["RVSR_pressure:0_m:mb"],
         name= "RVSR_pressure:0_m:mb",
         line=dict(color="blue"),
         connectgaps=False,
         opacity=0.5,
)
traceh =go.Scatter(
         x=flipLtd.index,
         y=flipLtd["FLIP_pressure:?:mb"],
         name= "FLIP_pressure:?:mb",
         line=dict(color="green"),
         connectgaps=False,
         opacity=0.5,
)
drawTraces(*[traceg, traceh])

In [102]:
tracea = go.Scatter(
         x=flipLtd.index,
         y=flipLtd["FLIP_hr_filtered_wind_speed:12.35_m:m/s"],
         name= "FLIP_hr_filtered_wind_speed:12.35_m:m/s",
         line=dict(color="blue"),
         connectgaps=False,
         opacity=0.5,
)
traceb =go.Scatter(
         x=flipLtd.index,
         y=flipLtd["FLIP_hr_filtered_wind_speed:15.76_m:m/s"],
         name= "FLIP_hr_filtered_wind_speed:15.76_m:m/s",
         line=dict(color="green"),
         connectgaps=False,
         opacity=0.5,
)
drawTraces(*[tracea, traceb])

In [103]:
tracee = go.Scatter(
         x=flipLtd.index,
         y=flipLtd["FLIP_hr_specific_humidity:12.35_m:g/Kg"],
         name= "FLIP_hr_specific_humidity:12.35_m:g/Kg",
         line=dict(color="blue"),
         connectgaps=False,
         opacity=0.5,
)
tracef =go.Scatter(
         x=flipLtd.index,
         y=flipLtd["FLIP_hr_specific_humidity:15.76_m:g/Kg"],
         name= "FLIP_hr_specific_humidity:15.76_m:g/Kg",
         line=dict(color="green"),
         connectgaps=False,
         opacity=0.5,
)
drawTraces(*[tracee, tracef])

In [104]:
tracec = go.Scatter(
         x=flipLtd.index,
         y=flipLtd["FLIP_hr_air_tmp:12.35_m:C"],
         name= "FLIP_hr_air_tmp:12.35_m:C",
         line=dict(color="blue"),
         connectgaps=False,
         opacity=0.5,
)
traced =go.Scatter(
         x=flipLtd.index,
         y=flipLtd["FLIP_hr_air_tmp:15.76_m:C"],
         name= "FLIP_hr_air_tmp:15.76_m:C",
         line=dict(color="green"),
         connectgaps=False,
         opacity=0.5,
)
traced2 =go.Scatter(
         x=flipLtd.index,
         y=flipLtd["FLIP_skin_temperature_at_saturation:0_m:C"],
         name= "FLIP_skin_temperature_at_saturation:0_m:C",
         line=dict(color="red"),
         connectgaps=False,
         opacity=0.5,
)
drawTraces(*[tracec, traced, traced2])

In [105]:
flipLtd = flip[keep_cols]
flipLtd = flipLtd.drop(['FLIP_proftime', 'index'], axis = 1)
print( flipLtd.columns)

Index(['Time', 'RVSR_pressure:0_m:mb',
       'RVSR_water_surface_temperature:-3.5_m:C',
       'FLIP_skin_temperature_at_saturation:0_m:C',
       'FLIP_specific_humidity:?:k/Kg', 'FLIP_pressure:?:mb',
       'FLIP_downwelling_lw_irradiance:?:W/m2',
       'FLIP_downwelling_sw_irradiance:?:W/m2', 'FLIP_wind_dir:?:degrees',
       'FLIP_hr_wind_speed:12.35_m:m/s',
       'FLIP_hr_filtered_wind_speed:12.35_m:m/s', 'FLIP_hr_air_tmp:12.35_m:C',
       'FLIP_hr_specific_humidity:12.35_m:g/Kg',
       'FLIP_hr_wind_speed:15.76_m:m/s',
       'FLIP_hr_filtered_wind_speed:15.76_m:m/s', 'FLIP_hr_air_tmp:15.76_m:C',
       'FLIP_hr_specific_humidity:15.76_m:g/Kg',
       'FLIP_friction_velocity:13.73_m:m/s',
       'FLIP_filtered_friction_velocity:13.73_m:m/s',
       'FLIP_wind_stress:13.73_m:N/m2',
       'FLIP_filtered_wind_stress:13.73_m:N/m2',
       'FLIP_turbulent_temp_scale:13.73_m:C',
       'FLIP_filtered_turbulent_temp_scale:13.73_m:C',
       'FLIP_turbulent_water_vapor_scale:13.73_